Copyright 2026 Ayush Yadav

Licensed under the Apache License, Version 2.0
http://www.apache.org/licenses/LICENSE-2.0

Convolutional Model

In [ ]:
from tensorflow.keras import layers, models, Input

def conv_unit(x, unit_id):
    """
    A single residual block for the ECG classifier.
    """
    shortcut = x
    s = str(unit_id)
    
    # First Convolution
    x = layers.Conv1D(filters=32, kernel_size=5, strides=1, 
                      padding='same', activation='relu', name=f'Conv1_{s}')(x)
    
    # Second Convolution
    x = layers.Conv1D(filters=32, kernel_size=5, strides=1, 
                      padding='same', activation=None, name=f'Conv2_{s}')(x)
    
    # Residual Connection: Add original input back
    x = layers.Add(name=f'ResidualSum_{s}')([x, shortcut])
    
    # Post-addition Activation and Max Pooling
    x = layers.Activation("relu", name=f'Act_{s}')(x)
    x = layers.MaxPooling1D(pool_size=5, strides=2, name=f'MaxPool_{s}')(x)
    
    return x

In [ ]:
from tensorflow.keras import layers, models, Input

def create_cnn_model(input_shape=(187, 1)):
    # 1. Define the input tensor
    inputs = Input(shape=input_shape, name='Input')
    
    # 2. Initial Projection
    x = layers.Conv1D(filters=32, kernel_size=5, strides=1, padding='same')(inputs)
    
    # 3. Residual Stack
    for i in range(5):
        # We assume conv_unit returns the transformed tensor 'x'
        x = conv_unit(x, i + 1)
        
    # 4. Dense Head
    x = layers.Flatten()(x)
    x = layers.Dense(32, name='FC1', activation='relu')(x)
    
    # 5. Output Layer (CORRECTED SYNTAX: No 'outputs=' keyword here)
    outputs = layers.Dense(5, name='Output', activation='softmax')(x)
    
    # 6. Build Model (Explicitly define inputs and outputs)
    model = models.Model(inputs=inputs, outputs=outputs)
    
    return model
model = create_cnn_model()
model.summary()

In [ ]:
# The input_shape must match the 3D shape (187, 1)
model = create_cnn_model(input_shape=(187, 1))

Estimator setup

In [ ]:
import tensorflow as tf

# 1. Define the Learning Rate Schedule
# Replaces: tf.train.exponential_decay
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=4000,
    decay_rate=0.5,
    staircase=True
)

# 2. Define the Optimizer with Clipping
# Replaces: tf.train.AdamOptimizer & gradient clipping
optimizer = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    clipnorm=10.0  # This handles the GRADIENT_NORM_THRESH
)

# 3. Compile the Model
# Replaces: EstimatorSpec and loss computation
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

Next run Training cells.